In [0]:
%pip install -U -qq langchain-databricks langchain==0.3.7 langchain_community==0.3.5 flashrank
dbutils.library.restartPython()

In [0]:
%sql

select * from flash.assessment.teradata_to_databricks

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS flash.codebuild;

In [0]:
%sql
CREATE TABLE flash.codebuild.teradata_to_databricks_out (
  script_name STRING,
  script_original STRING,
  converted_script STRING,
  complexity STRING,
  number_of_joins INT,
  comments STRING,
  script_type STRING,
  reason STRING,
  estimated_effort_days DOUBLE
)
TBLPROPERTIES (
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);


In [0]:
from langchain_databricks import ChatDatabricks
from langchain.agents import Tool, initialize_agent
from pyspark.sql import SparkSession
import re
import yaml

# Initialize Spark and LLM
spark = SparkSession.builder.getOrCreate()
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=7500)

# Extract number of joins
def extract_joins(script: str) -> int:
    return len(re.findall(r"\bjoin\b", script, flags=re.IGNORECASE))

# Extract only YAML from response
def extract_yaml_from_response(response_text: str) -> str:
    match = re.search(r"```y[a]?ml(.*?)```", response_text, re.DOTALL)
    return match.group(1).strip() if match else response_text.strip()

# Convert to PySpark
def convert_script_to_pyspark(script_content: str, script_name: str) -> str:
    prompt = f"""
            You are a data migration assistant. Convert the following Informatica mapping script ({script_name}) into PySpark DataFrame code.

            Script Content : {script_content}

            Instructions:
            - Focus on the logic contained within TRANSFORMATION blocks, especially EXPRESSION attributes from TRANSFORMFIELD nodes.
            - Use PySpark DataFrame API: functions like col(), when().otherwise(), lit(), isNull(), cast(), etc.
            - Preserve the logic and calculation flow.
            - Ignore metadata or structural definitions unrelated to transformation logic.
            - If applicable, chain withColumn() calls for field-level transformations.

            Return only the converted PySpark code inside triple backticks, like this:

            ```python
            # PySpark transformation logic
            df = df.withColumn("...")
            Do not include any additional text, comments, or explanation outside the code block.
        """
    response = llm.invoke(prompt)
    match = re.search(r"```(?:python)?(.*?)```", response.content, re.DOTALL)
    return match.group(1).strip() if match else response.content.strip()

# Assess converted script
def assess_converted_script(script: str, script_name: str) -> dict:
    prompt = f"""
Assess the following PySpark script for complexity:

Script Name: {script_name}
Script:
{script}

Return in YAML format:
complexity: <Low|Medium|High>
reason: "<Explanation>"
estimated_effort_days: <Number>
comments: "<Additional Notes>"
"""
    response = llm.invoke(prompt)
    return yaml.safe_load(extract_yaml_from_response(response.content))

# Write to Unity Catalog with MERGE
def write_conversion_output(script_name: str, original: str, converted: str, script_type: str, meta: dict, joins: int):
    df = spark.createDataFrame([{
        "script_name": script_name,
        "script_original": original,
        "converted_script": converted,
        "complexity": meta.get("complexity", "Unknown"),
        "number_of_joins": joins,
        "comments": meta.get("comments", ""),
        "script_type": script_type,
        "reason": meta.get("reason", ""),
        "estimated_effort_days": float(meta.get("estimated_effort_days", 0))
    }])

    df.createOrReplaceTempView("temp_codebuild")
    spark.sql("""
    MERGE INTO flash.codebuild.teradata_to_databricks_out AS target
    USING temp_codebuild AS source
    ON target.script_name = source.script_name AND target.script_type = source.script_type
    WHEN MATCHED THEN UPDATE SET
        target.script_original = source.script_original,
        target.converted_script = source.converted_script,
        target.complexity = source.complexity,
        target.number_of_joins = source.number_of_joins,
        target.comments = source.comments,
        target.reason = source.reason,
        target.estimated_effort_days = source.estimated_effort_days
    WHEN NOT MATCHED THEN INSERT *
    """)

# Full Conversion Pipeline
def convert_and_store_scripts(script_type_filter: str):
    df = spark.table("flash.assessment.teradata_to_databricks").filter(f"type = '{script_type_filter}'")
    rows = df.collect()
    if not rows:
        return f"No {script_type_filter} scripts to process."

    for row in rows:
        script_id = row["id"]
        original_script = row["script"]
        script_name = f"{script_type_filter}_script_{script_id}.txt"
        print(f"🚀 Converting: {script_name}")

        converted = convert_script_to_pyspark(original_script, script_name)
        meta = assess_converted_script(converted, script_name)
        joins = extract_joins(original_script)

        write_conversion_output(script_name, original_script, converted, script_type_filter, meta, joins)
    return f"✅ All {script_type_filter} scripts processed."

# LangChain Tool Wrappers
def convert_bteq_tool(_):
    return convert_and_store_scripts("bteq")

def convert_xml_tool(_):
    return convert_and_store_scripts("xml")

# Register tools
tools = [
    Tool(name="convert_bteq_scripts", func=convert_bteq_tool, description="Convert all BTEQ scripts to PySpark and store in Unity Catalog"),
    Tool(name="convert_xml_scripts", func=convert_xml_tool, description="Convert all Informatica XML jobs to PySpark and store in Unity Catalog")
]

# Initialize agent
agent = initialize_agent(tools=tools, llm=llm, agent="zero-shot-react-description", verbose=True)

# Entry point
def run_conversion_agent():
    print("\n🔄 BTEQ Conversion:")
    ##print(agent.run("Convert all BTEQ scripts to PySpark"))

    print("\n🔄 Informatica XML Conversion:")
    print(agent.run("Convert all Informatica XML jobs to PySpark"))

if __name__ == "__main__":
    run_conversion_agent()


In [0]:
%sql

select * from  flash.codebuild.teradata_to_databricks_out



In [0]:
from langchain_databricks import ChatDatabricks
from langchain.agents import Tool, initialize_agent
from pyspark.sql import SparkSession
import re
import yaml

# Initialize Spark and LLM
spark = SparkSession.builder.getOrCreate()
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2500)

# Extract number of joins
def extract_joins(script: str) -> int:
    return len(re.findall(r"\bjoin\b", script, flags=re.IGNORECASE))

# Extract only YAML from response
def extract_yaml_from_response(response_text: str) -> str:
    match = re.search(r"```y[a]?ml(.*?)```", response_text, re.DOTALL)
    return match.group(1).strip() if match else response_text.strip()

# Convert BTEQ to PySpark
def convert_bteq_to_pyspark(script_content: str, script_name: str) -> str:
    prompt = f"""
You are a migration assistant. Convert the following {script_name} BTEQ script to PySpark code using DataFrames.

Script:
{script_content}

Return only the converted PySpark code inside triple backticks.
"""
    response = llm.invoke(prompt)
    match = re.search(r"```(?:python)?(.*?)```", response.content, re.DOTALL)
    return match.group(1).strip() if match else response.content.strip()

# Convert Informatica XML to PySpark
def convert_xml_to_pyspark(script_content: str, script_name: str) -> str:
    prompt = f"""
You are a migration assistant. Convert the following {script_name} Informatica XML script to PySpark code using DataFrames.

Script:
{script_content}

Return only the converted PySpark code inside triple backticks.
"""
    response = llm.invoke(prompt)
    match = re.search(r"```(?:python)?(.*?)```", response.content, re.DOTALL)
    return match.group(1).strip() if match else response.content.strip()

# Assess converted script
def assess_converted_script(script: str, script_name: str) -> dict:
    prompt = f"""
Assess the following PySpark script for complexity:

Script Name: {script_name}
Script:
{script}

Return in YAML format:
complexity: <Low|Medium|High>
reason: "<Explanation>"
estimated_effort_days: <Number>
comments: "<Additional Notes>"
"""
    response = llm.invoke(prompt)
    return yaml.safe_load(extract_yaml_from_response(response.content))

# Write to Unity Catalog with MERGE
def write_conversion_output(script_name: str, original: str, converted: str, script_type: str, meta: dict, joins: int):
    df = spark.createDataFrame([{
        "script_name": script_name,
        "script_original": original,
        "converted_script": converted,
        "complexity": meta.get("complexity", "Unknown"),
        "number_of_joins": joins,
        "comments": meta.get("comments", ""),
        "script_type": script_type,
        "reason": meta.get("reason", ""),
        "estimated_effort_days": float(meta.get("estimated_effort_days", 0))
    }])

    df.createOrReplaceTempView("temp_codebuild")
    spark.sql("""
    MERGE INTO flash.codebuild.teradata_to_databricks_out AS target
    USING temp_codebuild AS source
    ON target.script_name = source.script_name AND target.script_type = source.script_type
    WHEN MATCHED THEN UPDATE SET
        target.script_original = source.script_original,
        target.converted_script = source.converted_script,
        target.complexity = source.complexity,
        target.number_of_joins = source.number_of_joins,
        target.comments = source.comments,
        target.reason = source.reason,
        target.estimated_effort_days = source.estimated_effort_days
    WHEN NOT MATCHED THEN INSERT *
    """)

# Convert and store BTEQ scripts
def convert_bteq_scripts():
    df = spark.table("flash.assessment.teradata_to_databricks").filter("type = 'bteq'")
    rows = df.collect()
    if not rows:
        return "No BTEQ scripts to process."

    for row in rows:
        script_id = row["id"]
        original_script = row["script"]
        script_name = f"bteq_script_{script_id}.txt"
        print(f"🚀 Converting: {script_name}")

        converted = convert_bteq_to_pyspark(original_script, script_name)
        meta = assess_converted_script(converted, script_name)
        joins = extract_joins(original_script)

        write_conversion_output(script_name, original_script, converted, "bteq", meta, joins)
    return "✅ All BTEQ scripts processed."

# Convert and store Informatica XML scripts
def convert_informatica_scripts():
    df = spark.table("flash.assessment.teradata_to_databricks").filter("type = 'xml'")
    rows = df.collect()
    if not rows:
        return "No Informatica XML scripts to process."

    for row in rows:
        script_id = row["id"]
        original_script = row["script"]
        script_name = f"xml_script_{script_id}.txt"
        print(f"🚀 Converting: {script_name}")

        converted = convert_xml_to_pyspark(original_script, script_name)
        meta = assess_converted_script(converted, script_name)
        joins = extract_joins(original_script)

        write_conversion_output(script_name, original_script, converted, "xml", meta, joins)
    return "✅ All Informatica XML scripts processed."

# LangChain Tool Wrappers
def convert_bteq_tool(_):
    return convert_bteq_scripts()

def convert_xml_tool(_):
    return convert_informatica_scripts()

# Register tools
tools = [
    Tool(name="convert_bteq_scripts", func=convert_bteq_tool, description="Convert all BTEQ scripts to PySpark and store in Unity Catalog"),
    Tool(name="convert_xml_scripts", func=convert_xml_tool, description="Convert all Informatica XML jobs to PySpark and store in Unity Catalog")
]

# Initialize agent
agent = initialize_agent(tools=tools, llm=llm, agent="zero-shot-react-description", verbose=True)

# Entry point
def run_conversion_agent():
    print("\n🔄 BTEQ Conversion:")
    print(agent.run("Convert all BTEQ scripts to PySpark"))

    print("\n🔄 Informatica XML Conversion:")
    #print(agent.run("Convert all Informatica XML jobs to PySpark"))

if __name__ == "__main__":
    run_conversion_agent()


In [0]:
%sql

select * from flash.codebuild.teradata_to_databricks_out